In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
#!/usr/bin/env python3
"""
Complete RIS Parser - Keeps ALL Records
Includes records without abstracts (conference proceedings, editorials, etc.)
"""
import pandas as pd
import re

def parse_ris_file(filepath):
    """Parse RIS format file - keeps ALL records"""
    
    print(f"\n{'='*80}")
    print(f"Parsing RIS file: {filepath}")
    print(f"{'='*80}")
    
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    
    # Split into individual records by ER tag
    records_raw = re.split(r'\nER  -\s*\n', content)
    
    records = []
    
    for idx, record_text in enumerate(records_raw):
        if not record_text.strip():
            continue
        
        # Dictionary to store this record's data
        record = {
            'Title': '',
            'Authors': [],
            'Abstract': '',
            'Year': '',
            'Journal': '',
            'DOI': '',
            'Volume': '',
            'Issue': '',
            'Pages': '',
            'Country': '',
            'Publication_Type': ''
        }
        
        lines = record_text.split('\n')
        current_field = None
        current_value = []
        
        for line in lines:
            # Check if line starts with a known tag
            if line.startswith('TY  - '):
                if current_field:
                    record[current_field] = ' '.join(current_value).strip()
                record['Publication_Type'] = line[6:].strip()
                current_field = None
                current_value = []
                
            elif line.startswith('TI  - '):
                if current_field:
                    record[current_field] = ' '.join(current_value).strip()
                current_field = 'Title'
                current_value = [line[6:].strip()]
                
            elif line.startswith('N2  - '):  # Abstract field
                if current_field:
                    record[current_field] = ' '.join(current_value).strip()
                current_field = 'Abstract'
                current_value = [line[6:].strip()]
                
            elif line.startswith('AU  - '):
                if current_field and current_field not in ['Authors']:
                    record[current_field] = ' '.join(current_value).strip()
                    current_field = None
                    current_value = []
                author = line[6:].strip()
                if author:
                    record['Authors'].append(author)
                    
            elif line.startswith('PY  - '):
                if current_field:
                    record[current_field] = ' '.join(current_value).strip()
                    current_field = None
                    current_value = []
                record['Year'] = line[6:].strip()
                
            elif line.startswith('JO  - '):
                if current_field:
                    record[current_field] = ' '.join(current_value).strip()
                    current_field = None
                    current_value = []
                record['Journal'] = line[6:].strip()
                
            elif line.startswith('DO  - '):
                if current_field:
                    record[current_field] = ' '.join(current_value).strip()
                    current_field = None
                    current_value = []
                record['DOI'] = line[6:].strip()
                
            elif line.startswith('VL  - '):
                if current_field:
                    record[current_field] = ' '.join(current_value).strip()
                    current_field = None
                    current_value = []
                record['Volume'] = line[6:].strip()
                
            elif line.startswith('IS  - '):
                if current_field:
                    record[current_field] = ' '.join(current_value).strip()
                    current_field = None
                    current_value = []
                record['Issue'] = line[6:].strip()
                
            elif line.startswith('SP  - '):
                if current_field:
                    record[current_field] = ' '.join(current_value).strip()
                    current_field = None
                    current_value = []
                record['Pages'] = line[6:].strip()
                
            elif line.startswith('CY  - '):
                if current_field:
                    record[current_field] = ' '.join(current_value).strip()
                    current_field = None
                    current_value = []
                record['Country'] = line[6:].strip()
                
            elif current_field and line.strip() and not line.startswith('  '):
                current_value.append(line.strip())
        
        # Save last field
        if current_field:
            record[current_field] = ' '.join(current_value).strip()
        
        # Clean up authors list
        record['Authors'] = '; '.join([a for a in record['Authors'] if a])
        
        # Keep ALL records that have at least a title
        if len(record['Title']) > 0:
            records.append(record)
    
    df = pd.DataFrame(records)
    
    # Add a column to flag records without abstracts
    df['Has_Abstract'] = df['Abstract'].str.len() > 0
    
    print(f"\n{'='*80}")
    print(f"PARSING RESULTS")
    print(f"{'='*80}")
    print(f"âœ“ Total records parsed: {len(df)}")
    print(f"  - With abstracts: {df['Has_Abstract'].sum()}")
    print(f"  - Without abstracts: {(~df['Has_Abstract']).sum()}")
    print(f"{'='*80}\n")
    
    return df


if __name__ == "__main__":
    # Parse the RIS file
    input_file = "/kaggle/input/enablingvoices478articles301025/review_594674_select_endnote_20251030230303.ris"
    df_complete = parse_ris_file(input_file)
    
    # Save complete version
    output_file = "/kaggle/working/complete_review_data.csv"
    df_complete.to_csv(output_file, index=False, encoding='utf-8')
    print(f"âœ“ Saved complete CSV with {len(df_complete)} records")
    print(f"  Output: {output_file}")
    
    # Show statistics
    print(f"\n{'='*80}")
    print("DATASET STATISTICS")
    print(f"{'='*80}")
    print(f"Total records: {len(df_complete)}")
    print(f"With abstracts: {df_complete['Has_Abstract'].sum()}")
    print(f"Without abstracts: {(~df_complete['Has_Abstract']).sum()}")
    print(f"Years range: {df_complete['Year'].min()} - {df_complete['Year'].max()}")
    print(f"Unique journals: {df_complete['Journal'].nunique()}")
    print(f"Records with DOI: {df_complete['DOI'].notna().sum()}")
    print(f"Publication types: {df_complete['Publication_Type'].value_counts().to_dict()}")
    
    # Show sample with and without abstracts
    print(f"\n{'='*80}")
    print("SAMPLE RECORDS WITH ABSTRACTS")
    print(f"{'='*80}")
    sample_with = df_complete[df_complete['Has_Abstract']].head(2)
    for i, row in sample_with.iterrows():
        print(f"\nRecord {i+1}:")
        print(f"  Title: {row['Title'][:100]}...")
        print(f"  Authors: {row['Authors'][:60]}...")
        print(f"  Year: {row['Year']}")
        print(f"  Abstract: {row['Abstract'][:100]}...")
    
    print(f"\n{'='*80}")
    print("SAMPLE RECORDS WITHOUT ABSTRACTS")
    print(f"{'='*80}")
    sample_without = df_complete[~df_complete['Has_Abstract']].head(2)
    for i, row in sample_without.iterrows():
        print(f"\nRecord {i+1}:")
        print(f"  Title: {row['Title'][:100]}")
        print(f"  Authors: {row['Authors'][:60]}")
        print(f"  Year: {row['Year']}")
        print(f"  Journal: {row['Journal'][:60]}")
        print(f"  [No abstract available]")


Parsing RIS file: /kaggle/input/enablingvoices478articles301025/review_594674_select_endnote_20251030230303.ris


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/enablingvoices478articles301025/review_594674_select_endnote_20251030230303.ris'